<a href="https://colab.research.google.com/github/poojithavakada7-hue/Movie_Recomendation/blob/main/Movie_Recomendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors

# Load dataset
df = pd.read_csv("movie_recommendation_dataset_3600.csv")

# Input features
features = [
    "genre",
    "language",
    "country",
    "release_year",
    "director",
    "lead_actor",
    "weather",

]

# Convert numeric columns
numeric_features = [
    "release_year",
]

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep records with movie titles
df = df.dropna(subset=["title"]).reset_index(drop=True)

# Prepare input data
X = df[features]

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), [c for c in features if c not in numeric_features])
])

# Train recommendation model
X_processed = preprocessor.fit_transform(X)

model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

model.fit(X_processed)

# Get user input
print("\nEnter Your Movie Preferences")

user_input = {}

for feature in features:
    value = input(f"Enter {feature}: ").strip()

    if feature in numeric_features:
        user_input[feature] = float(value) if value else np.nan
    else:
        user_input[feature] = value if value else np.nan

# Convert user input into DataFrame
input_df = pd.DataFrame([user_input], columns=features)

# Transform input using trained preprocessing pipeline
input_processed = preprocessor.transform(input_df)

# Find 5 most similar movies
distances, indices = model.kneighbors(
    input_processed,
    n_neighbors=min(5, len(df))
)

# Display only recommended movie titles
print("\nRecommended Movies:")

for rank, idx in enumerate(indices[0], start=1):
    print(f"{rank}. {df.iloc[idx]['title']}")


Enter Your Movie Preferences
Enter genre: thriller
Enter language: korean
Enter country: usa
Enter release_year: 2010
Enter director: david miller
Enter lead_actor: rahul
Enter weather: rainy

Recommended Movies:
1. Journey Home 24
2. Shadow Protocol
3. Secret Agent 62
4. Broken Compass
5. Lost in Mumbai 63


In [ ]:

# ------------------------------------------
# KNN MODEL EVALUATION WITH ACCURACY
# ------------------------------------------

K = 5

precision_scores = []
recall_scores = []

total_relevant_recommendations = 0
total_recommendations = 0

genres = (
    df["genre"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
    .to_numpy()
)

for i in range(len(df)):

    n = min(K + 1, len(df))

    distances, indices = model.kneighbors(
        X_processed[i],
        n_neighbors=n
    )

    # Remove the query movie itself
    recommended_indices = [
        idx for idx in indices[0] if idx != i
    ][:K]

    if not recommended_indices:
        continue

    # Relevant movies have the same genre
    relevant_movies = {
        idx for idx in range(len(df))
        if idx != i and genres[idx] == genres[i]
    }

    # Count relevant recommendations
    relevant_recommended = sum(
        idx in relevant_movies
        for idx in recommended_indices
    )

    # Precision@5
    precision = (
        relevant_recommended / len(recommended_indices)
    )

    # Recall@5
    recall = (
        relevant_recommended / len(relevant_movies)
        if relevant_movies else 0
    )

    precision_scores.append(precision)
    recall_scores.append(recall)

    # Accumulate for overall proxy accuracy
    total_relevant_recommendations += relevant_recommended
    total_recommendations += len(recommended_indices)


# Calculate evaluation metrics
precision_at_5 = np.mean(precision_scores) * 100
recall_at_5 = np.mean(recall_scores) * 100

# Accuracy (genre-match proxy)
accuracy = (
    total_relevant_recommendations / total_recommendations * 100
    if total_recommendations > 0 else 0
)

# F1-score
if precision_at_5 + recall_at_5 > 0:
    f1_score = (
        2 * precision_at_5 * recall_at_5
        / (precision_at_5 + recall_at_5)
    )
else:
    f1_score = 0


# Display results
print("\nKNN MODEL EVALUATION")
print("----------------------------")
print(f"Accuracy     : {accuracy:.2f}%")
print(f"Precision@5  : {precision_at_5:.2f}%")
print(f"Recall@5     : {recall_at_5:.2f}%")
print(f"F1-Score     : {f1_score:.2f}%")


KNN MODEL EVALUATION
----------------------------
Accuracy     : 55.79%
Precision@5  : 55.79%
Recall@5     : 0.96%
F1-Score     : 1.88%


In [ ]:
import pandas as pd

df = pd.read_csv("movie_recommendation_dataset_3600.csv")

print(df.columns.tolist())

['movie_id', 'title', 'genre', 'language', 'country', 'release_year', 'runtime_minutes', 'certificate', 'imdb_rating', 'vote_count', 'popularity_score', 'budget_usd', 'revenue_usd', 'director', 'lead_actor', 'user_age_rating', 'weather', 'director_favorite_color', 'data_entry_date']


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.svm import SVC
import pandas as pd

# Input features (re-declared for self-contained cell logic)
features = [
    "genre",
    "language",
    "country",
    "release_year",
    "director",
    "lead_actor",
    "weather",
]

# Numeric features (re-declared for self-contained cell logic)
numeric_features = [
    "release_year",
]

# Ensure 'release_year' is numeric in df before creating X for this cell's operations
# This assumes df is the global DataFrame and might have been reloaded.
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

def update_recommendations(imdb_rating_threshold):
    clear_output(wait=True) # Clear previous output for dynamic display

    # Work on a copy of the DataFrame to avoid modifying the global 'df' with the 'recommend' column
    temp_df = df.copy()

    # Create a 'recommend' column based on the new imdb_rating_threshold
    temp_df['recommend'] = (temp_df['imdb_rating'] >= imdb_rating_threshold).astype(int)

    # Target labels
    y = temp_df["recommend"]

    # Input features
    X = temp_df[features]

    # Preprocess movie features
    # Use 'transform' since 'preprocessor' is assumed to be already fitted from cell TdJNg9Uvg9Cp
    X_processed = preprocessor.transform(X)

    # Create SVM model
    model = SVC(kernel="linear", probability=True)

    # Train SVM
    model.fit(X_processed, y)

    # Predict recommendations
    predictions = model.predict(X_processed)

    # Display recommended movies
    recommended_movies = temp_df[predictions == 1]

    print(f"Recommended Movies (IMDb Rating Threshold: {imdb_rating_threshold:.1f}):")

    # Handle case where no movies are recommended with the given threshold
    if recommended_movies.empty:
        print("No movies recommended with this threshold. Try a lower threshold.")
    else:
        for title in recommended_movies["title"].head(5):
            print(title)

# Create the slider widget
imdb_slider = widgets.FloatSlider(
    value=7.0, # Initial value, matching previous default
    min=1.0,   # Minimum IMDb rating
    max=9.9,   # Maximum IMDb rating
    step=0.1,  # Step size
    description='IMDb Threshold:',
    continuous_update=False, # Only update when slider is released
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

# Link the slider to the update function
interactive_output = widgets.interactive_output(update_recommendations, {'imdb_rating_threshold': imdb_slider})

# Display the slider and the output
display(imdb_slider, interactive_output)

FloatSlider(value=7.0, continuous_update=False, description='IMDb Threshold:', max=9.9, min=1.0, readout_forma…

Output()

In [ ]:

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors

# ----------------------------------
# 1. LOAD DATASET
# ----------------------------------
df = pd.read_csv("movie_recommendation_dataset_3600.csv")

features = [
    "genre", "language", "country", "release_year",
    "director", "lead_actor", "weather"
]

numeric_features = ["release_year"]

for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["title"]).reset_index(drop=True)

# ----------------------------------
# 2. PREPROCESSING
# ----------------------------------
genre_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

other_cat_features = [
    "language", "country", "director",
    "lead_actor", "weather"
]

other_cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("genre", genre_pipeline, ["genre"]),
    ("cat", other_cat_pipeline, other_cat_features),
    ("num", numeric_pipeline, numeric_features)
], transformer_weights={
    "genre": 3.0,  # Increase genre importance
    "cat": 1.0,
    "num": 1.0
})

X = df[features]
X_processed = preprocessor.fit_transform(X)

# ----------------------------------
# 3. TRAIN KNN MODEL
# ----------------------------------
model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=min(6, len(df))
)

model.fit(X_processed)

print("Improved KNN model trained successfully!")

# ----------------------------------
# 4. EVALUATION
# ----------------------------------
K = 5

genres = (
    df["genre"].fillna("")
    .astype(str).str.lower().str.strip().to_numpy()
)

total_correct = 0
total_recommended = 0
precision_scores = []
recall_scores = []

for i in range(len(df)):

    n = min(K + 1, len(df))

    distances, indices = model.kneighbors(
        X_processed[i], n_neighbors=n
    )

    recommended = [
        idx for idx in indices[0] if idx != i
    ][:K]

    if not recommended:
        continue

    relevant = {
        idx for idx in range(len(df))
        if idx != i and genres[idx] == genres[i]
    }

    correct = sum(idx in relevant for idx in recommended)

    precision_scores.append(correct / len(recommended))
    recall_scores.append(
        correct / len(relevant) if relevant else 0
    )

    total_correct += correct
    total_recommended += len(recommended)

accuracy = (
    total_correct / total_recommended * 100
    if total_recommended else 0
)

precision = np.mean(precision_scores) * 100
recall = np.mean(recall_scores) * 100

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("\nIMPROVED KNN MODEL EVALUATION")
print("--------------------------------")
print(f"Genre-match Accuracy : {accuracy:.2f}%")
print(f"Precision@5          : {precision:.2f}%")
print(f"Recall@5             : {recall:.2f}%")
print(f"F1-Score             : {f1:.2f}%")

# ----------------------------------
# 5. GET USER PREFERENCES
# ----------------------------------
print("\nEnter Your Movie Preferences")

user_input = {}

for feature in features:
    value = input(f"Enter {feature}: ").strip()

    if feature in numeric_features:
        user_input[feature] = float(value) if value else np.nan
    else:
        user_input[feature] = value if value else np.nan

input_df = pd.DataFrame([user_input], columns=features)

input_processed = preprocessor.transform(input_df)

distances, indices = model.kneighbors(
    input_processed,
    n_neighbors=min(5, len(df))
)

print("\nRecommended Movies:")

for rank, idx in enumerate(indices[0], start=1):
    print(f"{rank}. {df.iloc[idx]['title']}")

Improved KNN model trained successfully!

IMPROVED KNN MODEL EVALUATION
--------------------------------
Genre-match Accuracy : 96.82%
Precision@5          : 96.82%
Recall@5             : 1.66%
F1-Score             : 3.26%

Enter Your Movie Preferences
Enter genre: thirller
Enter language: hindi
Enter country: usa
Enter release_year: 2010
Enter director: shiv
Enter lead_actor: rajiv
Enter weather: cool

Recommended Movies:
1. Journey Home 24
2. Shadow Protocol
3. Secret Agent 62
4. Broken Compass
5. Lost in Mumbai 63
